# hexbot · Colab quickstart

Train and watch a Hex Connect-6 bot from a free Colab T4 GPU in about five minutes.

1. **Runtime → Change runtime type → GPU**
2. **Run each cell top to bottom.**

This is the five-minute tour. For deeper guided walkthroughs see:

- [01 · Game basics](01_game_basics.ipynb): placing stones, win detection, reading the board.
- [02 · Analysis tools and writing a bot](02_analysis_and_bots.ipynb): threats, alpha-beta, your first bot.
- [03 · Training Orca (AlphaZero-style)](03_training_orca.ipynb): same content as below, but expanded with manifest inspection, AutoTuner dry-run, and Zoo packaging.

Full docs: [Wiki](https://github.com/Saiki77/hexbot-building-framework/wiki).

## 1. Install

Pulls the latest `hexbot` from PyPI plus `tensorboard` so we can watch loss / ELO live.

In [ ]:
!pip install --quiet hexbot tensorboard

## 2. Verify the install

Plays one game between the bundled Orca checkpoint and the heuristic bot. You should see Orca win. The shipped checkpoint is early-stage (~iteration 65) but already stronger than the rule-based opponent.

In [ ]:
from hexbot import HexGame, Bot, Arena

result = Arena(Bot.orca(), Bot.heuristic(), num_games=1).play()
print(result)

## 3. Watch Orca think

Before training, peek at how Orca actually picks moves: MCTS guided by the policy head. The top entries are the moves MCTS expanded the most.

In [ ]:
from hexbot import mcts_search

g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

result = mcts_search(g, sims=200)
print(f"best move: {result['best_move']}")
for move, visits in result['top_moves'][:5]:
    print(f"  {move}  visits={visits}")

## 4. Train

Twenty self-play iterations with the `colab-t4` hardware profile. Takes about three minutes on a T4. Loss, ELO, and per-iteration timings are written to TensorBoard under `runs/<timestamp>/`; the manifest captures CLI args + config + git sha so the run is reproducible.

In [ ]:
!python -m orca.train --profile=colab-t4 --iterations 20 --tensorboard

## 5. Inspect training metrics

Loads TensorBoard inline. Look for `loss/total` falling and `elo/current` climbing.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## 6. Test the newly trained bot

Reload from the latest checkpoint and play 5 games against the heuristic. With only 20 iterations the gain over the bundled Orca will be modest; the win rate vs heuristic is what to focus on.

In [ ]:
import glob, torch
from hexbot import Bot, Arena

ckpts = sorted(glob.glob('hex_checkpoint_*.pt'),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
latest = ckpts[-1]
meta = torch.load(latest, weights_only=False)['_hexbot_meta']
print(f"latest: {latest}  (iter={meta['iter']}, arch={meta['arch']})")

my_bot = Bot.from_checkpoint(latest)
print(Arena(my_bot, Bot.heuristic(), num_games=5).play())

## What next

Pick the path that matches what you want to do:

| If you want to ... | Open |
|---|---|
| Learn the game engine API end-to-end | [01 · Game basics](01_game_basics.ipynb) |
| Write your own (non-neural) bot | [02 · Analysis tools and writing a bot](02_analysis_and_bots.ipynb) |
| Train Orca seriously (100+ iterations) | [03 · Training Orca](03_training_orca.ipynb) + [Training Guide](https://github.com/Saiki77/hexbot-building-framework/wiki/Training-Guide) |
| Share or download community bots | [Evaluation and Sharing](https://github.com/Saiki77/hexbot-building-framework/wiki/Evaluation-and-Sharing) |
| Play your bot on hexo.did.science | [Playing Online](https://github.com/Saiki77/hexbot-building-framework/wiki/Playing-Online) |